# 📅 Day 02: Segment Tree — Range Queries with Updates

---

## When Prefix Sum Isn't Enough

Prefix sum gives O(1) range sum queries — but what if the array **changes**? Recomputing prefix takes O(n) per update. For `q` queries with updates, that's O(n*q).

**Segment Tree** gives **O(log n) for BOTH query and update**. It's a binary tree where:
- **Leaves** = individual array elements
- **Internal nodes** = aggregation of their range (sum, min, max, etc.)

### Visual: How It Looks

```
  Array: [1, 3, 5, 7]
  
            [16]          ← root = sum of all
           /    \\
         [4]    [12]      ← sum of halves
        /  \\   /  \\
      [1]  [3] [5]  [7]  ← leaves = original array
  
  Query sum(1,2): need [3]+[5] = walk tree → O(log n)
  Update idx=0 to 2: update leaf, walk up fixing parents → O(log n)
```

### Array Implementation (Interview-Friendly)

Store the tree in a flat array of size `2n`:
- `tree[n..2n-1]` = leaves (original elements)
- `tree[i]` has children `tree[2i]` and `tree[2i+1]`
- `tree[1]` = root = answer for the entire range

```
╔═══════════════════════════════════════════════════════════════════════╗
║  WHEN TO USE SEGMENT TREE                                          ║
║                                                                     ║
║  ✅ Range queries (sum/min/max) + point updates                    ║
║  ✅ Both operations needed in O(log n)                             ║
║  ❌ Don't use if array is static (prefix sum is simpler)           ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🎯 Problem: Range Sum Query - Mutable (LC #307)

Support `update(idx, val)` and `sumRange(left, right)` efficiently.

### Step-by-Step Trace

```
  Build from [1, 3, 5]:
  n=3, tree has 6 slots: [_, _, _, 1, 3, 5]
                                   ↑ leaves at indices 3,4,5
  Fill internal: tree[2]=3+5=8, tree[1]=1+8=9
  tree = [0, 9, 8, 1, 3, 5]
  
  query(0, 2): L=3, R=6
    L=3 (odd) → add tree[3]=1, L=4
    L=2, R=3 → L=2 (even), R=3 (odd) → add tree[2]=8, R=2
    L >= R → done. Result = 1+8 = 9 ✓
  
  update(1, 2): pos=4, tree[4]=2
    pos=2: tree[2]=tree[4]+tree[5]=2+5=7
    pos=1: tree[1]=tree[2]+tree[3]=7+1=8
```

In [ ]:
class SegmentTree:
    """
    Array-based segment tree for range sum queries.
    tree[1] = root (total sum), tree[n..2n-1] = leaves.
    """
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (2 * self.n)
        
        # Fill leaves
        for i in range(self.n):
            self.tree[self.n + i] = nums[i]
        
        # Fill internal nodes bottom-up
        for i in range(self.n - 1, 0, -1):
            self.tree[i] = self.tree[2*i] + self.tree[2*i+1]
    
    def update(self, idx, val):
        """Set nums[idx] = val. Propagate up. O(log n)"""
        pos = idx + self.n
        self.tree[pos] = val
        while pos > 1:
            pos //= 2
            self.tree[pos] = self.tree[2*pos] + self.tree[2*pos+1]
    
    def query(self, left, right):
        """
        Sum of nums[left..right] inclusive. O(log n).
        Walk from leaves toward root, collecting partial sums.
        """
        result = 0
        left += self.n
        right += self.n + 1
        while left < right:
            if left & 1:   # Left is a right-child → include it
                result += self.tree[left]
                left += 1
            if right & 1:  # Right is a right-child → include its sibling
                right -= 1
                result += self.tree[right]
            left //= 2
            right //= 2
        return result

st = SegmentTree([1, 3, 5])
print(st.query(0, 2))  # 9 (1+3+5)
st.update(1, 2)        # [1, 2, 5]
print(st.query(0, 2))  # 8
print(st.query(1, 2))  # 7

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  SEGMENT TREE PATTERNS                                             ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  Range sum + updates       → Segment tree (or Fenwick)            ║
║  Range min/max + updates   → Segment tree ONLY (Fenwick can't)    ║
║  Count in range            → Segment tree on value space          ║
║  Calendar overlap          → Segment tree + lazy propagation      ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 307 | Range Sum Query - Mutable | Medium | Basic segment tree |
| 315 | Count of Smaller After Self | Hard | Segment tree on values |
| 729 | My Calendar I | Medium | Overlap detection |
| 731 | My Calendar II | Medium | Lazy propagation |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Segment tree: O(log n) query + O(log n) update               │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Array trick: leaves at [n..2n-1], root at [1]                 │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Only for MUTABLE arrays — prefix sum for static ones          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Works for any associative operation: sum, min, max, GCD       │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Fenwick Tree (BIT)